In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import random
import numpy as np
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, get_scheduler, AutoTokenizer
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Split Data

In [4]:
# 1. Persiapan Data (Simulasi)
# Ganti bagian ini dengan pd.read_csv("nama_file_anda.csv") jika data sudah siap
data = pd.read_csv("/content/drive/MyDrive/Kuliah/Semester 6/Jurnal/dataset/Komentar_Youtube_bersih_berlabel.csv")
df = pd.DataFrame(data)

# Pisahkan fitur (X) dan target/label (y)
X = df['Teks']
y = df['Label']

# 2. Langkah Pertama: Pisahkan Data Uji (Test)
# Kita ambil 15% untuk Test, sisanya 85% akan menjadi data latih sementara (Train + Val)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y  # Menjaga proporsi kelas (Positif/Negatif) tetap seimbang
)

# 3. Langkah Kedua: Pisahkan sisa data (X_temp) menjadi Train dan Validation
# Kita ingin Validation menjadi 15% dari TOTAL data awal.
# Karena X_temp adalah 85% dari total, maka persentase pembagiannya adalah: 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=(0.15 / 0.85),
    random_state=42,
    stratify=y_temp
)

# 4. (Opsional) Gabungkan kembali menjadi DataFrame agar rapi
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df = pd.DataFrame({'text': X_val, 'label': y_val})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

# 5. Tampilkan Hasil Pembagian
print("=== HASIL PEMBAGIAN DATA ===")
print(f"Total Data Awal: {len(df)}")
print(f"Data Latih (Train)  : {len(train_df)} data ({(len(train_df)/len(df))*100:.0f}%)")
print(f"Data Validasi (Val) : {len(val_df)} data ({(len(val_df)/len(df))*100:.0f}%)")
print(f"Data Uji (Test)     : {len(test_df)} data ({(len(test_df)/len(df))*100:.0f}%)")

=== HASIL PEMBAGIAN DATA ===
Total Data Awal: 10990
Data Latih (Train)  : 7692 data (70%)
Data Validasi (Val) : 1649 data (15%)
Data Uji (Test)     : 1649 data (15%)


# Tokenizing

In [5]:
# 1. Definisikan path model dari Hugging Face yang kamu gunakan
# (Sesuaikan path ini jika kamu menggunakan versi spesifik lainnya)
model_paths = {
    "IndoBERT": "indobenchmark/indobert-base-p1",
    "IndoRoBERTa": "cahya/roberta-base-indonesian-522M",
    "IndoELECTRA": "ChristopherA08/IndoELECTRA"
}

# 2. Inisialisasi dictionary untuk menyimpan ketiga tokenizer
tokenizers = {}
for model_name, path in model_paths.items():
    print(f"Mengunduh tokenizer untuk {model_name}...")
    tokenizers[model_name] = AutoTokenizer.from_pretrained(path)

# 3. Fungsi Tokenisasi
def tokenize_data(texts, tokenizer, max_length=128):
    return tokenizer(
        texts.tolist(),
        padding='max_length', # Menyamakan panjang kalimat (menambah angka 0 jika kalimat pendek)
        truncation=True,      # Memotong kalimat jika melebihi max_length
        max_length=max_length,
        return_tensors='pt'   # 'pt' untuk PyTorch, ubah jadi 'tf' jika pakai TensorFlow
    )

Mengunduh tokenizer untuk IndoBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Mengunduh tokenizer untuk IndoELECTRA...


config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [6]:
# ==========================================
# Tokenizing
# ==========================================

# 4. Lakukan tokenisasi untuk Masing-masing Model dan Masing-masing Split
print("\nMelakukan Tokenisasi untuk Train, Val, dan Test...")
tokenized_inputs = {}

for model_name, tokenizer in tokenizers.items():
    print(f"-> Memproses {model_name}...")
    tokenized_inputs[model_name] = {
        # Ganti 'text' dengan nama kolom Anda
        'train': tokenize_data(train_df['text'], tokenizer),
        'val': tokenize_data(val_df['text'], tokenizer),
        'test': tokenize_data(test_df['text'], tokenizer)
    }

print("Selesai! Data siap digunakan untuk PyTorch/TensorFlow.")

# Contoh cara memanggil hasilnya:
# input_ids_train_indobert = tokenized_inputs["IndoBERT"]['train']['input_ids']


Melakukan Tokenisasi untuk Train, Val, dan Test...
-> Memproses IndoBERT...
-> Memproses IndoRoBERTa...
-> Memproses IndoELECTRA...
Selesai! Data siap digunakan untuk PyTorch/TensorFlow.


In [7]:
print("Kolom yang tersedia di dalam data tokenized (contoh IndoBERT, train split):")
# Ambil salah satu model dan split untuk contoh
example_tokenized_data = tokenized_inputs["IndoBERT"]['train']

# Cetak kunci-kunci (kolom) yang ada
for key in example_tokenized_data.keys():
    print(f"- {key}")

# Jika ingin melihat bentuk datanya, contoh:
print("\nBentuk data 'input_ids' untuk IndoBERT (train):")
print(example_tokenized_data['input_ids'].shape)

Kolom yang tersedia di dalam data tokenized (contoh IndoBERT, train split):
- input_ids
- token_type_ids
- attention_mask

Bentuk data 'input_ids' untuk IndoBERT (train):
torch.Size([7692, 128])


In [8]:
input_ids_train_indobert = tokenized_inputs["IndoBERT"]['train']['input_ids']
print (input_ids_train_indobert)

input_ids_train_indoroberta = tokenized_inputs["IndoRoBERTa"]['train']['input_ids']
print (input_ids_train_indoroberta)

input_ids_train_indoelectra = tokenized_inputs["IndoELECTRA"]['train']['input_ids']
print (input_ids_train_indoelectra)

tensor([[    2,  5254,   599,  ...,     0,     0,     0],
        [    2,    92,  8601,  ...,     0,     0,     0],
        [    2,  8003,    34,  ...,     0,     0,     0],
        ...,
        [    2, 10248,   294,  ...,     0,     0,     0],
        [    2,  1248,   300,  ...,     0,     0,     0],
        [    2, 12792,  3906,  ...,     0,     0,     0]])
tensor([[    0,   397,  1194,  ...,     1,     1,     1],
        [    0,   884, 10604,  ...,     1,     1,     1],
        [    0, 32661,   292,  ...,     1,     1,     1],
        ...,
        [    0, 14027,  7254,  ...,     1,     1,     1],
        [    0, 31084, 16671,  ...,     1,     1,     1],
        [    0,   387,   836,  ...,     1,     1,     1]])
tensor([[    2,  5555,  2030,  ...,     0,     0,     0],
        [    2,  1565,  8142,  ...,     0,     0,     0],
        [    2,  8456,  1532,  ...,     0,     0,     0],
        ...,
        [    2,  7518,  1805,  ...,     0,     0,     0],
        [    2,  2704,  1821,  

# Training

In [10]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

In [25]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [26]:
from sklearn.preprocessing import LabelEncoder

NUM_LABELS = len(train_df['label'].unique())
print(NUM_LABELS)

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_val   = le.transform(y_val)
y_test  = le.transform(y_test)

print(le.classes_)

3
['Negative' 'Neutral' 'Positive']


In [34]:
# Dataset wrapper
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# DataLoader seed
def get_dataloader(dataset, batch_size=16):
    g = torch.Generator()
    g.manual_seed(42)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=g
    )

# Train
def train_epoch(model, dataloader, optimizer, scheduler):
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# Eval
def eval_model(model, dataloader):
    model.eval()

    preds, labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            logits = outputs.logits

            pred = torch.argmax(logits, dim=1)

            preds.extend(pred.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    precision = precision_score(labels, preds, average="weighted")
    recall = recall_score(labels, preds, average="weighted")

    return acc, f1, precision, recall, preds, labels

# Filter inputs function for models that don't use token_type_ids (e.g., RoBERTa, ELECTRA)
def filter_inputs(data):
    return {
        "input_ids": data["input_ids"],
        "attention_mask": data["attention_mask"]
    }

## Model IndoBert

In [28]:
model_name = "indobenchmark/indobert-base-p1"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS
).to(device)

train_dataset = CustomDataset(tokenized_inputs["IndoBERT"]['train'], y_train)
val_dataset   = CustomDataset(tokenized_inputs["IndoBERT"]['val'], y_val)

train_loader = get_dataloader(train_dataset)
val_loader   = DataLoader(val_dataset, batch_size=16)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)

scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:
for epoch in range(num_epochs):
    print(f"\n[IndoBERT] Epoch {epoch+1}")

    loss = train_epoch(model, train_loader, optimizer, scheduler)
    acc, f1, precision, recall, _, _ = eval_model(model, val_loader)

    print(f"Loss: {loss:.4f}")
    print(f"Acc: {acc:.4f} | F1: {f1:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")


[IndoBERT] Epoch 1


  0%|          | 0/481 [00:00<?, ?it/s]/tmp/ipykernel_1321/2507538496.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
100%|██████████| 481/481 [02:56<00:00,  2.72it/s]


Loss: 0.4184
Acc: 0.8575 | F1: 0.8416
Precision: 0.8495 | Recall: 0.8575

[IndoBERT] Epoch 2


  0%|          | 0/481 [00:00<?, ?it/s]/tmp/ipykernel_1321/2507538496.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
100%|██████████| 481/481 [02:57<00:00,  2.72it/s]


Loss: 0.1961
Acc: 0.8642 | F1: 0.8659
Precision: 0.8684 | Recall: 0.8642

[IndoBERT] Epoch 3


  0%|          | 0/481 [00:00<?, ?it/s]/tmp/ipykernel_1321/2507538496.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
100%|██████████| 481/481 [02:56<00:00,  2.72it/s]


Loss: 0.0664
Acc: 0.8684 | F1: 0.8659
Precision: 0.8642 | Recall: 0.8684


## Model IndoRoBERTa

In [36]:
model_name = "flax-community/indonesian-roberta-base"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS
).to(device)

# Apply filter_inputs to remove token_type_ids for IndoRoBERTa
train_dataset = CustomDataset(
    filter_inputs(tokenized_inputs["IndoRoBERTa"]['train']),
    y_train
)
val_dataset   = CustomDataset(
    filter_inputs(tokenized_inputs["IndoRoBERTa"]['val']),
    y_val
)

train_loader = get_dataloader(train_dataset)
val_loader   = DataLoader(val_dataset, batch_size=16)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)

scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: flax-community/indonesian-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [32]:
for epoch in range(num_epochs):
    print(f"\n[IndoRoBERTa] Epoch {epoch+1}")

    loss = train_epoch(model, train_loader, optimizer, scheduler)
    acc, f1, precision, recall, _, _ = eval_model(model, val_loader)

    print(f"Loss: {loss:.4f}")
    print(f"Acc: {acc:.4f} | F1: {f1:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")


[IndoRoBERTa] Epoch 1


  0%|          | 0/481 [00:00<?, ?it/s]/tmp/ipykernel_1321/2507538496.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
  0%|          | 0/481 [00:00<?, ?it/s]


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Model IndoElectra

In [ ]:
model_name = "ChristopherA08/IndoELECTRA" # Corrected model path from model_paths["IndoELECTRA"]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS
).to(device)

# ⚠️ filter input (hindari CUDA error) - Function moved to kDVdJXKEtlon
# def filter_inputs(data):
#     return {
#         "input_ids": data["input_ids"],
#         "attention_mask": data["attention_mask"]
#     }

train_dataset = CustomDataset(
    filter_inputs(tokenized_inputs["IndoELECTRA"]['train']),
    y_train
)

val_dataset = CustomDataset(
    filter_inputs(tokenized_inputs["IndoELECTRA"]['val']),
    y_val
)

train_loader = get_dataloader(train_dataset)
val_loader   = DataLoader(val_dataset, batch_size=16)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)

scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

In [ ]:
for epoch in range(num_epochs):
    print(f"\n[IndoELECTRA] Epoch {epoch+1}")

    loss = train_epoch(model, train_loader, optimizer, scheduler)
    acc, f1, precision, recall, _, _ = eval_model(model, val_loader)

    print(f"Loss: {loss:.4f}")
    print(f"Acc: {acc:.4f} | F1: {f1:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")

# COBA 2

In [ ]:
# --- 1. Define Hyperparameters and Set Seed for Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Hyperparameters
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# --- 2. Label Mapping (re-using definitions from previous cells) ---
# Assuming df, y_train, y_val, y_test are available from previous cells
unique_labels = sorted(df['Label'].unique())
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for i, label in enumerate(unique_labels)}
num_labels = len(unique_labels)
print(f"Label mapping: {label_to_id}")
print(f"Total label: {num_labels}")

y_train_encoded = [label_to_id[label] for label in y_train]
y_val_encoded = [label_to_id[label] for label in y_val]
y_test_encoded = [label_to_id[label] for label in y_test]

# --- 3. Custom Dataset Class (re-using definition from previous cells) ---
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # tokenized_inputs already contains torch.Tensor, so no need to re-tensorize
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long) # Ensure labels are long type for CrossEntropyLoss
        return item

    def __len__(self):
        return len(self.labels)

# --- 4. Metric Computation Function (adapted for manual loop) ---
def compute_metrics_manual(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

Label mapping: {'Negative': 0, 'Neutral': 1, 'Positive': 2}
Total label: 3


In [ ]:
# Set CUDA_LAUNCH_BLOCKING to 1 for synchronous CUDA operations
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
print("CUDA_LAUNCH_BLOCKING has been set to 1. Run your training code now.")
print("Remember to restart your runtime or set it back to '0' for optimal performance after debugging.")

CUDA_LAUNCH_BLOCKING has been set to 1. Run your training code now.
Remember to restart your runtime or set it back to '0' for optimal performance after debugging.


In [ ]:
# --- 5. Training and Evaluation Loop for each model (Manual Implementation) ---
results = {}

for model_name, path in model_paths.items():
    print(f"\n{'='*50}")
    print(f"Mulai melatih dan mengevaluasi model: {model_name} (Manual Loop)")
    print(f"{'='*50}")

    # Load Model and move to device
    model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=num_labels)
    model.to(device)

    # Create Datasets and DataLoaders
    train_dataset = CustomDataset(tokenized_inputs[model_name]['train'], y_train_encoded)
    val_dataset = CustomDataset(tokenized_inputs[model_name]['val'], y_val_encoded)
    test_dataset = CustomDataset(tokenized_inputs[model_name]['test'], y_test_encoded)

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Optimizer and Loss Function
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    best_val_f1 = -1.0 # Track best F1 for potential early stopping or best model saving

    from tqdm.auto import tqdm # Import tqdm

    for epoch in range(NUM_EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
        # --- Training Phase ---
        model.train()
        total_train_loss = 0
        train_preds_list = []
        train_labels_list = []

        # Wrap train_dataloader with tqdm
        for batch_idx, batch in enumerate(tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}")):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': batch['labels'].to(device)}
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(device)

            optimizer.zero_grad()
            outputs = model(**kwargs)
            loss = outputs.loss
            logits = outputs.logits

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            train_preds_list.extend(torch.argmax(logits, axis=1).cpu().numpy())
            train_labels_list.extend(batch['labels'].cpu().numpy())

            # Removed batch printing for cleaner tqdm output

        avg_train_loss = total_train_loss / len(train_dataloader)
        train_metrics = compute_metrics_manual(train_labels_list, train_preds_list)
        print(f"  Training Results - Loss: {avg_train_loss:.4f}, Accuracy: {train_metrics['accuracy']:.4f}, F1: {train_metrics['f1']:.4f}")

        # --- Validation Phase ---
        model.eval()
        total_val_loss = 0
        val_preds_list = []
        val_labels_list = []

        with torch.no_grad():
            # Wrap val_dataloader with tqdm
            for batch_idx, batch in enumerate(tqdm(val_dataloader, desc=f"Validation Epoch {epoch+1}")):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': batch['labels'].to(device)}
                if 'token_type_ids' in batch:
                    kwargs['token_type_ids'] = batch['token_type_ids'].to(device)

                outputs = model(**kwargs)
                loss = outputs.loss
                logits = outputs.logits

                total_val_loss += loss.item()
                val_preds_list.extend(torch.argmax(logits, axis=1).cpu().numpy())
                val_labels_list.extend(batch['labels'].cpu().numpy())

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_metrics = compute_metrics_manual(val_labels_list, val_preds_list)
        print(f"  Validation Results - Loss: {avg_val_loss:.4f}, Accuracy: {val_metrics['accuracy']:.4f}, F1: {val_metrics['f1']:.4f}, Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}")

        # Optional: Track best model based on validation F1
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            print(f"  New best validation F1: {best_val_f1:.4f}")
            # In a real scenario, you might save `model.state_dict()` here.

    print(f"Training for {model_name} finished.")

    # --- Save the trained model ---
    model_save_path = f"/content/drive/MyDrive/Kuliah/Semester 6/Jurnal/Model/{model_name.lower()}_model.pt"
    torch.save(model.state_dict(), model_save_path)
    print(f"Model {model_name} disimpan ke: {model_save_path}")

    # --- Final Evaluation on Test Set ---
    print(f"\nMelakukan evaluasi pada test set untuk {model_name}...")
    model.eval()
    test_preds_list = []
    test_labels_list = []
    total_test_loss = 0

    with torch.no_grad():
        # Wrap test_dataloader with tqdm for evaluation
        for batch_idx, batch in enumerate(tqdm(test_dataloader, desc=f"Evaluating {model_name}")):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': batch['labels'].to(device)}
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(device)

            outputs = model(**kwargs)
            loss = outputs.loss
            logits = outputs.logits

            total_test_loss += loss.item()
            test_preds_list.extend(torch.argmax(logits, axis=1).cpu().numpy())
            test_labels_list.extend(batch['labels'].cpu().numpy())

    avg_test_loss = total_test_loss / len(test_dataloader)
    test_metrics = compute_metrics_manual(test_labels_list, test_preds_list)
    results[model_name] = {
        'test_loss': avg_test_loss,
        'test_accuracy': test_metrics['accuracy'],
        'test_f1': test_metrics['f1'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall']
    }
    print(f"Hasil Evaluasi {model_name} pada Test Set: {results[model_name]}")


Mulai melatih dan mengevaluasi model: IndoBERT (Manual Loop)


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

ElectraForSequenceClassification LOAD REPORT from: ChristopherA08/IndoELECTRA
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the che


--- Epoch 1/3 ---


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Training Epoch 1:   0%|          | 0/481 [00:00<?, ?it/s]

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
print("\n\n=========== RINGKASAN HASIL EVALUASI SEMUA MODEL (Manual Loop) ==========")
for model_name, res in results.items():
    print(f"\nModel: {model_name}")
    for metric, value in res.items():
        print(f"  {metric}: {value:.4f}")



=========== RINGKASAN HASIL EVALUASI SEMUA MODEL (Manual Loop) ==========

Model: IndoBERT
  test_loss: 0.6299
  test_accuracy: 0.8454
  test_f1: 0.8258
  test_precision: 0.8313
  test_recall: 0.8454

Model: IndoRoBERTa
  test_loss: 0.6024
  test_accuracy: 0.8229
  test_f1: 0.7947
  test_precision: 0.8006
  test_recall: 0.8229


In [ ]:
# --- 5. Training and Evaluation Loop for each model (Manual Implementation) ---
results = {}

for model_name, path in model_paths.items():
    print(f"\n{'='*50}")
    print(f"Mulai melatih dan mengevaluasi model: {model_name} (Manual Loop)")
    print(f"{'='*50}")

    # Load Model and move to device
    model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=num_labels)
    model.to(device)

    # Create Datasets and DataLoaders
    train_dataset = CustomDataset(tokenized_inputs[model_name]['train'], y_train_encoded)
    val_dataset = CustomDataset(tokenized_inputs[model_name]['val'], y_val_encoded)
    test_dataset = CustomDataset(tokenized_inputs[model_name]['test'], y_test_encoded)

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Optimizer and Loss Function
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    best_val_f1 = -1.0 # Track best F1 for potential early stopping or best model saving

    from tqdm.auto import tqdm # Import tqdm

    for epoch in range(NUM_EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
        # --- Training Phase ---
        model.train()
        total_train_loss = 0
        train_preds_list = []
        train_labels_list = []

        # Wrap train_dataloader with tqdm
        for batch_idx, batch in enumerate(tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}")):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': batch['labels'].to(device)}
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(device)

            optimizer.zero_grad()
            outputs = model(**kwargs)
            loss = outputs.loss
            logits = outputs.logits

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            train_preds_list.extend(torch.argmax(logits, axis=1).cpu().numpy())
            train_labels_list.extend(batch['labels'].cpu().numpy())

            # Removed batch printing for cleaner tqdm output

        avg_train_loss = total_train_loss / len(train_dataloader)
        train_metrics = compute_metrics_manual(train_labels_list, train_preds_list)
        print(f"  Training Results - Loss: {avg_train_loss:.4f}, Accuracy: {train_metrics['accuracy']:.4f}, F1: {train_metrics['f1']:.4f}")

        # --- Validation Phase ---
        model.eval()
        total_val_loss = 0
        val_preds_list = []
        val_labels_list = []

        with torch.no_grad():
            # Wrap val_dataloader with tqdm
            for batch_idx, batch in enumerate(tqdm(val_dataloader, desc=f"Validation Epoch {epoch+1}")):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': batch['labels'].to(device)}
                if 'token_type_ids' in batch:
                    kwargs['token_type_ids'] = batch['token_type_ids'].to(device)

                outputs = model(**kwargs)
                loss = outputs.loss
                logits = outputs.logits

                total_val_loss += loss.item()
                val_preds_list.extend(torch.argmax(logits, axis=1).cpu().numpy())
                val_labels_list.extend(batch['labels'].cpu().numpy())

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_metrics = compute_metrics_manual(val_labels_list, val_preds_list)
        print(f"  Validation Results - Loss: {avg_val_loss:.4f}, Accuracy: {val_metrics['accuracy']:.4f}, F1: {val_metrics['f1']:.4f}, Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}")

        # Optional: Track best model based on validation F1
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            print(f"  New best validation F1: {best_val_f1:.4f}")
            # In a real scenario, you might save `model.state_dict()` here.

    print(f"Training for {model_name} finished.")

    # --- Final Evaluation on Test Set ---
    print(f"\nMelakukan evaluasi pada test set untuk {model_name}...")
    model.eval()
    test_preds_list = []
    test_labels_list = []
    total_test_loss = 0

    with torch.no_grad():
        # Wrap test_dataloader with tqdm for evaluation
        for batch_idx, batch in enumerate(tqdm(test_dataloader, desc=f"Evaluating {model_name}")):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': batch['labels'].to(device)}
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(device)

            outputs = model(**kwargs)
            loss = outputs.loss
            logits = outputs.logits

            total_test_loss += loss.item()
            test_preds_list.extend(torch.argmax(logits, axis=1).cpu().numpy())
            test_labels_list.extend(batch['labels'].cpu().numpy())

    avg_test_loss = total_test_loss / len(test_dataloader)
    test_metrics = compute_metrics_manual(test_labels_list, test_preds_list)
    results[model_name] = {
        'test_loss': avg_test_loss,
        'test_accuracy': test_metrics['accuracy'],
        'test_f1': test_metrics['f1'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall']
    }
    print(f"Hasil Evaluasi {model_name} pada Test Set: {results[model_name]}")
